# Branch Smoke Checks: APH/PPH Split

This notebook does quick, rough validation checks for branch changes around the APH/PPH split and hemorrhage logic.

Checks covered:
1. APH and PPH incidence are in the ballpark of artifact expectations
2. Only severe hemorrhage cases die from hemorrhage
3. Misoprostol affects postpartum hemorrhage incidence
4. HemoglobinRiskEffect: low hemoglobin simulants have higher APH/PPH incidence
5. Exploratory anemia burden check stratified by hemorrhage severity

In [9]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd

import vivarium_gates_mncnh
from vivarium import Artifact, InteractiveContext
from vivarium.framework.configuration import build_model_specification

from vivarium_gates_mncnh.constants.data_keys import MATERNAL_HEMORRHAGE
from vivarium_gates_mncnh.constants.data_values import (
    ANEMIA_THRESHOLDS,
    COLUMNS,
    HEMORRHAGE_SEVERITY,
    SIMULATION_EVENT_NAMES,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [10]:
# Build a small interactive simulation for quick checks
model_spec_path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
base_spec = build_model_specification(model_spec_path)

DRAW = int(base_spec.configuration.input_data.input_draw_number)
POP_SIZE = 60_000

SIMULATION_STEPS = [
    SIMULATION_EVENT_NAMES.FIRST_TRIMESTER_ANC,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_SCREENING,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_INTERVENTION,
    SIMULATION_EVENT_NAMES.LATER_PREGNANCY_VISIT_TIMING,
    SIMULATION_EVENT_NAMES.ULTRASOUND,
    SIMULATION_EVENT_NAMES.ANTEPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.DELIVERY_FACILITY,
    SIMULATION_EVENT_NAMES.AZITHROMYCIN_ACCESS,
    SIMULATION_EVENT_NAMES.MISOPROSTOL_ACCESS,
    SIMULATION_EVENT_NAMES.CPAP_ACCESS,
    SIMULATION_EVENT_NAMES.ACS_ACCESS,
    SIMULATION_EVENT_NAMES.ANTIBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.PROBIOTICS_ACCESS,
    SIMULATION_EVENT_NAMES.OBSTRUCTED_LABOR,
    SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE,
    SIMULATION_EVENT_NAMES.MATERNAL_SEPSIS,
    SIMULATION_EVENT_NAMES.RESIDUAL_MATERNAL_DISORDERS,
    SIMULATION_EVENT_NAMES.ABORTION_MISCARRIAGE_ECTOPIC_PREGNANCY,
    SIMULATION_EVENT_NAMES.MORTALITY,
    SIMULATION_EVENT_NAMES.EARLY_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.LATE_NEONATAL_MORTALITY,
    SIMULATION_EVENT_NAMES.POSTPARTUM_DEPRESSION,
]

STEP_MAPPER = {name: i + 1 for i, name in enumerate(SIMULATION_STEPS)}

def make_spec(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    spec = deepcopy(base_spec)
    spec.configuration.population.population_size = pop_size
    spec.configuration.intervention.scenario = scenario
    return spec

def make_sim(scenario: str = 'baseline', pop_size: int = POP_SIZE):
    return InteractiveContext(make_spec(scenario=scenario, pop_size=pop_size))

def run_to_step(sim: InteractiveContext, step_name: str):
    sim.take_steps(STEP_MAPPER[step_name])
    return sim

artifact = Artifact(base_spec.configuration.input_data.artifact_path)
draw_col = f'draw_{DRAW}'
print('Using artifact:', base_spec.configuration.input_data.artifact_path)
print('Using draw column:', draw_col)
print('Population size:', POP_SIZE)

Using artifact: /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf
Using draw column: draw_60
Population size: 60000


## 1) APH/PPH incidence and severity vs artifact (rough)

In [11]:
def get_art_by_age(art_key: str, sim_year: int, sim_location: str, sim_sex: str = 'Female') -> pd.DataFrame:
    art = artifact.load(art_key).reset_index()

    # Filter progressively; if a filter wipes everything out, keep previous data
    cur = art.copy()
    if 'location' in cur.columns:
        next_df = cur[cur['location'] == sim_location]
        if not next_df.empty:
            cur = next_df
    if 'sex' in cur.columns:
        next_df = cur[cur['sex'] == sim_sex]
        if not next_df.empty:
            cur = next_df
    if {'year_start', 'year_end'}.issubset(cur.columns):
        next_df = cur[(cur['year_start'] <= sim_year) & (sim_year < cur['year_end'])]
        if not next_df.empty:
            cur = next_df

    grouped = cur.groupby(['age_start', 'age_end'], as_index=False)[draw_col].mean()
    return grouped.sort_values(['age_start', 'age_end']).reset_index(drop=True)

def map_ages_to_art_values(ages: pd.Series, art_by_age: pd.DataFrame) -> pd.Series:
    intervals = pd.IntervalIndex.from_arrays(
        art_by_age['age_start'].astype(float),
        art_by_age['age_end'].astype(float),
        closed='left',
    )
    idx = intervals.get_indexer(ages.astype(float))
    out = pd.Series(np.nan, index=ages.index, dtype=float)
    matched = idx >= 0
    if matched.any():
        out.loc[matched] = art_by_age.iloc[idx[matched]][draw_col].to_numpy()
    return out

def weighted_incidence_from_artifact(
    pop: pd.DataFrame,
    art_key: str,
    eligible_mask: pd.Series,
    sim_year: int,
    sim_location: str,
    sim_sex: str = 'Female',
) -> float:
    art_by_age = get_art_by_age(art_key, sim_year=sim_year, sim_location=sim_location, sim_sex=sim_sex)
    eligible_ages = pop.loc[eligible_mask, COLUMNS.MOTHER_AGE]
    values = map_ages_to_art_values(eligible_ages, art_by_age)
    matched = values.notna()
    return float(values[matched].mean()) if matched.any() else np.nan

sim_year = int(base_spec.configuration.time.start.year)

# Single run for both APH and PPH checks (up to PPH step)
sim = make_sim('baseline')
run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
pop = sim.get_population([
    COLUMNS.MOTHER_AGE,
    COLUMNS.LOCATION,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

sim_location = pop[COLUMNS.LOCATION].mode().iloc[0]

# Denominators must match component eligibility rules
aph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_eligible = pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

# Sim incidence
aph_cases = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool)
pph_cases = pop[COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool)

aph_incidence_sim = float(aph_cases[aph_eligible].mean()) if int(aph_eligible.sum()) else np.nan
pph_incidence_sim = float(pph_cases[pph_eligible].mean()) if int(pph_eligible.sum()) else np.nan

# Artifact incidence weighted to simulated eligible age composition
aph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.APH_INCIDENCE_RISK,
    aph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)
pph_incidence_art = weighted_incidence_from_artifact(
    pop,
    MATERNAL_HEMORRHAGE.PPH_INCIDENCE_RISK,
    pph_eligible,
    sim_year=sim_year,
    sim_location=sim_location,
)

check1 = pd.DataFrame([
    {
        'cause': 'antepartum_hemorrhage',
        'sim_incidence': aph_incidence_sim,
        'artifact_incidence': aph_incidence_art,
        'incidence_ratio_sim_over_artifact': aph_incidence_sim / aph_incidence_art if aph_incidence_art else np.nan,
    },
    {
        'cause': 'postpartum_hemorrhage',
        'sim_incidence': pph_incidence_sim,
        'artifact_incidence': pph_incidence_art,
        'incidence_ratio_sim_over_artifact': pph_incidence_sim / pph_incidence_art if pph_incidence_art else np.nan,
    },
])

check1

2026-05-14 14:14:58.722 | INFO     | simulation_7-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-05-14 14:14:58.723 | INFO     | simulation_7-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-05-14 14:14:58.724 | INFO     | simulation_7-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-05-14 14:15:26.854 | WARNING  | simulation_7-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:15:26.854 | WARNING  | simulation_7-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:15:26.959 | WARNING  | simulation_7-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-05-14 14:15:26.960 | WARNING  | simulation_7-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-05-14 14:15:26.960 | WARNING  | simulation_7-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-05-14 14:15:26.960 | WARNING  | simulation_7-lookup_table_manager:85 - 

2026-05-14 14:15:28.917 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-05-14 14:15:38.972 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-05-14 14:15:39.788 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-05-14 14:15:40.960 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-05-14 14:15:51.577 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-05-14 14:16:03.767 | INFO     | simulation_7 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-05-14 14:16:04.317 | WARNING  | simulation_7-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-05-14 14:16:04.395 | INFO     | simulation_7 - vivarium.framewor

,cause,sim_incidence,artifact_incidence,incidence_ratio_sim_over_artifact
0,antepartum_hemorrhage,0.048850,0.048107,1.015448
1,postpartum_hemorrhage,0.099419,0.099275,1.001453


## 2) Only severe hemorrhage cases die from hemorrhage

In [12]:
sim_mort = make_sim('baseline')
run_to_step(sim_mort, SIMULATION_EVENT_NAMES.MORTALITY)
mort_pop = sim_mort.get_population([
    COLUMNS.MOTHER_CAUSE_OF_DEATH,
    COLUMNS.ANTEPARTUM_HEMORRHAGE_SEVERITY,
    COLUMNS.POSTPARTUM_HEMORRHAGE_SEVERITY,
])

hemo_cod = [COLUMNS.ANTEPARTUM_HEMORRHAGE, COLUMNS.POSTPARTUM_HEMORRHAGE]
dead_hemo = mort_pop[mort_pop[COLUMNS.MOTHER_CAUSE_OF_DEATH].isin(hemo_cod)]

bad_aph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.ANTEPARTUM_HEMORRHAGE)
    & (dead_hemo[COLUMNS.ANTEPARTUM_HEMORRHAGE_SEVERITY] != HEMORRHAGE_SEVERITY.SEVERE)
]
bad_pph = dead_hemo[
    (dead_hemo[COLUMNS.MOTHER_CAUSE_OF_DEATH] == COLUMNS.POSTPARTUM_HEMORRHAGE)
    & (dead_hemo[COLUMNS.POSTPARTUM_HEMORRHAGE_SEVERITY] != HEMORRHAGE_SEVERITY.SEVERE)
]

assert len(bad_aph) == 0, f'{len(bad_aph)} non-severe APH deaths found'
assert len(bad_pph) == 0, f'{len(bad_pph)} non-severe PPH deaths found'
print(f'PASSED: all {len(dead_hemo)} hemorrhage deaths were severe')

2026-05-14 14:16:15.867 | INFO     | simulation_8-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-05-14 14:16:15.868 | INFO     | simulation_8-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-05-14 14:16:15.869 | INFO     | simulation_8-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-05-14 14:16:43.039 | WARNING  | simulation_8-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:16:43.040 | WARNING  | simulation_8-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:16:43.147 | WARNING  | simulation_8-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-05-14 14:16:43.147 | WARNING  | simulation_8-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-05-14 14:16:43.148 | WARNING  | simulation_8-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-05-14 14:16:43.148 | WARNING  | simulation_8-lookup_table_manager:85 - 

2026-05-14 14:16:45.087 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-05-14 14:16:54.920 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-05-14 14:16:55.733 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-05-14 14:16:56.894 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-05-14 14:17:07.483 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-05-14 14:17:19.634 | INFO     | simulation_8 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-05-14 14:17:20.181 | WARNING  | simulation_8-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-05-14 14:17:20.260 | INFO     | simulation_8 - vivarium.framewor

## 3) Misoprostol affects postpartum hemorrhage incidence

In [13]:
def pph_summary_for_scenario(scenario: str) -> pd.DataFrame:
    sim = make_sim(scenario)
    run_to_step(sim, SIMULATION_EVENT_NAMES.POSTPARTUM_HEMORRHAGE)
    pop = sim.get_population([
        COLUMNS.POSTPARTUM_HEMORRHAGE,
        COLUMNS.MISOPROSTOL_AVAILABLE,
        COLUMNS.DELIVERY_FACILITY_TYPE,
    ])

    out = []
    overall = float(pop[COLUMNS.POSTPARTUM_HEMORRHAGE].mean())
    out.append({'scenario': scenario, 'group': 'overall', 'pph_incidence': overall, 'n': len(pop)})

    for value in [True, False]:
        sub = pop[pop[COLUMNS.MISOPROSTOL_AVAILABLE] == value]
        if len(sub) == 0:
            continue
        out.append({
            'scenario': scenario,
            'group': f'misoprostol_available={value}',
            'pph_incidence': float(sub[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(sub),
        })

    home = pop[pop[COLUMNS.DELIVERY_FACILITY_TYPE] == 'home']
    if len(home):
        out.append({
            'scenario': scenario,
            'group': 'home_only',
            'pph_incidence': float(home[COLUMNS.POSTPARTUM_HEMORRHAGE].mean()),
            'n': len(home),
        })

    return pd.DataFrame(out)

baseline_pph = pph_summary_for_scenario('baseline')
miso_vv_pph = pph_summary_for_scenario('misoprostol_vv')

check4 = pd.concat([baseline_pph, miso_vv_pph], ignore_index=True)
check4

2026-05-14 14:18:57.546 | INFO     | simulation_9-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-05-14 14:18:57.547 | INFO     | simulation_9-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-05-14 14:18:57.548 | INFO     | simulation_9-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-05-14 14:19:25.020 | WARNING  | simulation_9-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:19:25.021 | WARNING  | simulation_9-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:19:25.126 | WARNING  | simulation_9-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-05-14 14:19:25.126 | WARNING  | simulation_9-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-05-14 14:19:25.127 | WARNING  | simulation_9-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-05-14 14:19:25.127 | WARNING  | simulation_9-lookup_table_manager:85 - 

2026-05-14 14:19:27.063 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-05-14 14:19:36.891 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-05-14 14:19:37.714 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-05-14 14:19:38.882 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-05-14 14:19:49.467 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-05-14 14:20:01.448 | INFO     | simulation_9 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-05-14 14:20:02.001 | WARNING  | simulation_9-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-05-14 14:20:02.084 | INFO     | simulation_9 - vivarium.framewor

/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-05-14 14:20:30.123 | WARNING  | simulation_10-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:20:30.124 | WARNING  | simulation_10-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:20:30.230 | WARNING  | simulation_10-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-05-14 14:20:30.230 | WARNING  | simulation_10-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-05-14 14:20:30.231 | WARNING  | simulation_10-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-05-14 14:20:30.231 | WARNING  | simulation_10-lookup_table_manager

2026-05-14 14:20:32.174 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-05-14 14:20:41.989 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-05-14 14:20:42.799 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-05-14 14:20:43.956 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-05-14 14:20:54.729 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-05-14 14:21:06.743 | INFO     | simulation_10 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-05-14 14:21:07.295 | WARNING  | simulation_10-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-05-14 14:21:07.373 | INFO     | simulation_10 - vivarium.

,scenario,group,pph_incidence,n
0,baseline,overall,0.056167,60000
1,baseline,misoprostol_available=False,0.056167,60000
2,baseline,home_only,0.102035,14887
3,misoprostol_vv,overall,0.053883,60000
4,misoprostol_vv,misoprostol_available=True,0.064964,4110
5,misoprostol_vv,misoprostol_available=False,0.053069,55890
6,misoprostol_vv,home_only,0.092833,14887


In [15]:
# Directional check: compare home-birth PPH incidence across scenarios.
# Misoprostol is only available to home births so that is the right comparison group.
baseline_home = float(check4.query("scenario == 'baseline' and group == 'home_only'")['pph_incidence'].iloc[0])
miso_home = float(check4.query("scenario == 'misoprostol_vv' and group == 'home_only'")['pph_incidence'].iloc[0])

pd.DataFrame([
    {'metric': 'baseline_home_birth_pph_incidence', 'value': baseline_home},
    {'metric': 'misoprostol_vv_home_birth_pph_incidence', 'value': miso_home},
    {'metric': 'directional_pass', 'value': miso_home < baseline_home},
])

,metric,value
0,baseline_home_birth_pph_incidence,0.102035
1,misoprostol_vv_home_birth_pph_incidence,0.092833
2,directional_pass,True


## 4) HemoglobinRiskEffect modifies APH/PPH incidence

Higher hemoglobin is protective (RR < 1 at high exposure). Expected direction: low-hemoglobin tertile has higher APH and PPH incidence than the high-hemoglobin tertile.

In [16]:
# Reuse sim from check 1 (already run to PPH step)
hgb_pop = sim.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.PREGNANCY_OUTCOME,
    COLUMNS.ANTEPARTUM_HEMORRHAGE,
    COLUMNS.POSTPARTUM_HEMORRHAGE,
])

aph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME] != 'invalid'
pph_elig = hgb_pop[COLUMNS.PREGNANCY_OUTCOME].isin(['live_birth', 'stillbirth'])

hgb_pop['hgb_tertile'] = pd.qcut(hgb_pop[COLUMNS.HEMOGLOBIN_EXPOSURE], q=3, labels=['low', 'mid', 'high'])

rows = []
for tertile in ['low', 'mid', 'high']:
    t_mask = hgb_pop['hgb_tertile'] == tertile
    rows.append({
        'hgb_tertile': tertile,
        'aph_incidence': float(hgb_pop.loc[t_mask & aph_elig, COLUMNS.ANTEPARTUM_HEMORRHAGE].astype(bool).mean()),
        'pph_incidence': float(hgb_pop.loc[t_mask & pph_elig, COLUMNS.POSTPARTUM_HEMORRHAGE].astype(bool).mean()),
    })

check4_hgb = pd.DataFrame(rows)

low_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'aph_incidence'].iloc[0]
high_aph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'aph_incidence'].iloc[0]
low_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'low', 'pph_incidence'].iloc[0]
high_pph = check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'pph_incidence'].iloc[0]

check4_hgb['directional_pass'] = None
check4_hgb.loc[check4_hgb['hgb_tertile'] == 'high', 'directional_pass'] = (low_aph > high_aph) and (low_pph > high_pph)

check4_hgb

,hgb_tertile,aph_incidence,pph_incidence,directional_pass
0,low,0.07640,0.155122,None
1,mid,0.03515,0.075262,None
2,high,0.03500,0.071244,True


## 5) Exploratory anemia check stratified by hemorrhage severity

This is a provisional check (not a strict test): compare average anemia disability weight by highest hemorrhage severity at the end of the run.
Expected direction: higher burden for severe > moderate > none.

In [17]:
def anemia_status_from_hgb(hgb: pd.Series) -> pd.Series:
    return (
        pd.cut(
            hgb,
            bins=[-np.inf] + ANEMIA_THRESHOLDS,
            labels=['severe', 'moderate', 'mild'],
            right=False,
        )
        .astype('object')
        .fillna('not_anemic')
    )

anemia_dw_map = {
    'severe': 0.149,
    'moderate': 0.052,
    'mild': 0.004,
    'not_anemic': 0.0,
}

sim_anemia = make_sim('baseline')
run_to_step(sim_anemia, SIMULATION_EVENT_NAMES.POSTPARTUM_DEPRESSION)
pop = sim_anemia.get_population([
    COLUMNS.HEMOGLOBIN_EXPOSURE,
    COLUMNS.ANTEPARTUM_HEMORRHAGE_SEVERITY,
    COLUMNS.POSTPARTUM_HEMORRHAGE_SEVERITY,
])

severity_rank = {HEMORRHAGE_SEVERITY.NONE: 0, HEMORRHAGE_SEVERITY.MODERATE: 1, HEMORRHAGE_SEVERITY.SEVERE: 2}
aph_rank = pop[COLUMNS.ANTEPARTUM_HEMORRHAGE_SEVERITY].map(severity_rank).fillna(0).astype(int)
pph_rank = pop[COLUMNS.POSTPARTUM_HEMORRHAGE_SEVERITY].map(severity_rank).fillna(0).astype(int)
max_rank = np.maximum(aph_rank.values, pph_rank.values)
rank_to_label = {0: 'none', 1: 'moderate', 2: 'severe'}
pop['max_hemorrhage_severity'] = pd.Series(max_rank, index=pop.index).map(rank_to_label)

pop['anemia_status'] = anemia_status_from_hgb(pop[COLUMNS.HEMOGLOBIN_EXPOSURE])
pop['anemia_dw'] = pop['anemia_status'].map(anemia_dw_map)

severity_summary = (
    pop.groupby('max_hemorrhage_severity')
    .agg(
        n=('anemia_dw', 'size'),
        mean_anemia_dw=('anemia_dw', 'mean'),
        severe_anemia_share=('anemia_status', lambda x: (x == 'severe').mean()),
        moderate_or_worse_share=('anemia_status', lambda x: x.isin(['severe', 'moderate']).mean()),
    )
    .sort_index()
)

severity_summary

2026-05-14 14:21:59.633 | INFO     | simulation_11-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-05-14 14:21:59.634 | INFO     | simulation_11-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-05-14 14:21:59.635 | INFO     | simulation_11-artifact_manager:82 - Artifact additional filter terms are None.


/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_tables = list(self.data.groupby(list(self.categorical_parameters)))
/ihme/homes/hjafari/miniconda3/envs/vivarium_gates_mncnh_simulation/lib/python3.11/site-pack

2026-05-14 14:22:27.757 | WARNING  | simulation_11-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:22:27.772 | WARNING  | simulation_11-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-05-14 14:22:28.050 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-05-14 14:22:28.051 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-05-14 14:22:28.051 | WARNING  | simulation_11-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-05-14 14:22:28.052 | WARNING  | simulation_11-lookup_table_manager

2026-05-14 14:22:30.142 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-01 00:00:00
2026-05-14 14:22:39.991 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-02 00:00:00
2026-05-14 14:22:40.793 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-03 00:00:00
2026-05-14 14:22:41.929 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-04 00:00:00
2026-05-14 14:22:52.534 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-05 00:00:00
2026-05-14 14:23:04.697 | INFO     | simulation_11 - vivarium.framework.engine:280 - 2025-01-06 00:00:00
2026-05-14 14:23:05.232 | WARNING  | simulation_11-population_manager:747 - The 'antepartum_hemorrhage.incidence_risk.paf' attribute pipeline returned a pd.Series with a different name 'value'. For the column being added to the population state table, we will use 'antepartum_hemorrhage.incidence_risk.paf'.
2026-05-14 14:23:05.311 | INFO     | simulation_11 - vivarium.

,n,mean_anemia_dw,severe_anemia_share,moderate_or_worse_share
max_hemorrhage_severity,,,,
moderate,5092,0.026951,0.075216,0.367636
none,53908,0.011517,0.019255,0.175002
severe,1000,0.026901,0.081000,0.356000
